# English Version — Module 2: Full-Text Source Download & Selective Extraction (`src/downloader`)

## Overview & Module Architecture

The `src/downloader` package downloads raw LaTeX source archives for every `(arxiv_id, version)` pair from arXiv, inspects and unpacks them **entirely in memory**, and persists only the text files required by downstream parsing and reference extraction (`.tex`, `.bib`, and conditional `.bbl`). Heavy binary assets (figures, PDFs, build artifacts) never touch the disk.

| Module | Responsibility |
| :--- | :--- |
| `config.py` | System constants: arXiv endpoints, allowed/conditional extensions, RAM & file size caps, retry limits, circuit-breaker thresholds, default paths. |
| `index.py` | Loads and validates `data/input/papers_index.json` (`{arxiv_id: ["v1", "v2", ...]}`), enforcing new-style IDs (`YYMM.NNNNN`) and valid `vN` tags. |
| `client.py` | Thread-safe HTTP client (`FastArxivClient`) with per-thread Keep-Alive sessions, global rate-slot pacing, adaptive cooldown, PDF-only early abort, and retry logic. |
| `extractor.py` | In-memory archive inspection and selective extraction (`.tar`, `.tar.gz`, streamed `.tar.gz`, `.zip`, single plain/gzipped `.tex`/`.bib`), path sanitization, collision renaming, conditional `.bbl` keeping, and atomic cleanup on error. |
| `utils.py` | Dependency-free helpers: TeX/Bib/HTML sniffing, comment-aware `\bibitem` detection, path sanitization, byte/duration formatting, OS-level Peak RSS measurement. |
| `pipeline.py` | Per-paper work units: concurrent version fetching, in-memory extraction to `papers/<YYMM-NNNNN>/<vN>/`, attempt counting, status merging, and byte-total aggregation. |
| `runner.py` | Core orchestrator (`_engine`) shared by both batch mode (`download_corpus`) and streaming mode (`download_stream`), managing the two-tier thread pools, atomic checkpoints, per-version resume, signal handling, and HTTP 403 circuit breaker. |
| `audit.py` | Post-crawl verification (`run_audit`) to detect and recover missing or transiently failed versions in bounded parallel batches. |
| `stats.py` | Corpus-wide storage metrics, atomic `download_summary.json` persistence, and formatted console report. |
| `cli.py` / `__main__.py` | CLI entry point (`python -m src.downloader`). |

## 2.1. Version-Level Source Download Flow, Rate Limiting & Fault Tolerance (`client.py`, `index.py`)

### Input Validation (`index.py`)
`load_papers_index()` reads `papers_index.json` without modifying it and strictly validates every entry:
- `arxiv_id` must match `^\d{4}\.\d{4,5}$` (old-style IDs such as `hep-th/9901001` are rejected because `aid.replace('.', '-')` would create unexpected nested directories).
- Each version list must be non-empty and match `^v\d+$`.

### HTTP Client & Global Rate Pacing (`client.py`)
`FastArxivClient` uses a **thread-local `requests.Session`** (`HTTPAdapter(pool_connections=2, pool_maxsize=2)` with `Connection: keep-alive`) and a fixed, honest `User-Agent` (with optional `--contact-email`). Even with multiple concurrent download threads (`DEFAULT_CONNECTIONS = 8`), all requests pass through a single global pacing gate `_wait_rate_slot()`:

```python
# src/downloader/client.py
def _wait_rate_slot(self) -> int:
    while True:
        if self.stop_event is not None and self.stop_event.is_set():
            raise InterruptedError('Pipeline stopped.')
        with self._rate_lock:
            now = time.monotonic()
            with self._cooldown_lock:
                pacing = self.delay - (now - self._last_req_ts) if self._last_req_ts is not None else 0.0
                wait = max(0.0, self._cooldown_until - now, pacing)
            if self.shared_cooldown is not None:
                wait = max(wait, self.shared_cooldown.remaining())
            if wait <= 0:
                idx = self._counter
                self._counter += 1
                self._last_req_ts = now
                return idx
        self._sleep(min(wait, 2.0))
```

Key reliability mechanisms in `fetch_version()`:
1. **Endpoint Fallback:** Tries `https://export.arxiv.org/src/{id}v{n}` first and alternates with `https://export.arxiv.org/e-print/{id}v{n}` on retries (up to `MAX_FETCH_ATTEMPTS = 7`).
2. **Adaptive Cooldown:** On HTTP `429` or `403`, `_note_throttle()` sets `_cooldown_until` to `max(min(25, 10 + 5 * attempt), Retry-After)` (capped at `MAX_RETRY_AFTER_SEC = 300s`), multiplies `delay` by `1.25x` (capped at `max(MAX_ADAPTIVE_DELAY, 2 * base_delay)`), and publishes `429` cooldowns to `shared_cooldown` so other arXiv modules pause together. After 25 consecutive clean responses, `_note_success()` relaxes `delay` by `0.8x` back toward `base_delay` (`DEFAULT_DELAY = 0.6s`).
3. **HTTP 403 Detection:** If HTTP `403` repeats `MAX_403_ATTEMPTS = 2` times on a version, `BlockedError` is raised immediately without further retries.
4. **HTTP 404 Distinction:** Returns a sentinel `_NOT_FOUND_PAYLOAD` instead of retrying, keeping 404s distinguishable from genuine PDF-only submissions.
5. **Streaming Body Validation & Early PDF Abort (`_read_body`):** Reads in 64 KiB chunks, enforces `MAX_ARCHIVE_BYTES = 500 MiB` (raising non-retryable `ArchiveTooLarge`), **immediately aborts on the first chunk if it starts with `b"%PDF-"`** (avoiding downloading full PDFs), and retries (`_BadPayload`) on empty bodies, HTML error pages (`looks_like_html`), or truncated transfers (`len(body) != Content-Length`).

## 2.2. Selective Extraction Mechanism & Path Safety (`extractor.py`, `utils.py`)

All archive inspection happens in RAM via `io.BytesIO`; only filtered text files are written to `data/output/papers/<YYMM-NNNNN>/<vN>/`.

### Filtering Rules & Conditional `.bbl` Retention
- **Always kept (`ALLOWED_EXTENSIONS`):** `.tex` and `.bib` files (case-insensitive).
- **Purged in memory:** Images (`.png`, `.jpg`, `.eps`, `.pdf`), build artifacts (`.aux`, `.log`, `.out`, `.toc`), macOS metadata (`__MACOSX`, `._*`), empty/whitespace-only files, and disguised binaries (containing `\x00` in the first 8 KiB or binary magic prefixes).
- **Conditional `.bbl` (`CONDITIONAL_EXTENSIONS`):** `.bbl` files are buffered in memory (`pending_bbl`) while scanning the archive and written **only if** the version has at least one `.tex` file, **zero `.bib` files**, and **no uncommented `\bibitem`** inside any kept `.tex` file (`has_inline_bibitem` ignores `\bibitem` after `%`). This preserves bibliography data for reference extraction when `.bbl` is the sole source:

```python
# src/downloader/extractor.py
if tex > 0 and bib == 0 and not has_bibitem:
    skipped.extend(oversize_bbl)  # a missing .bbl matters only now
    for name, rel, content in pending_bbl:
        final = _unique_rel(seen, rel)
        original = PurePosixPath(name.replace("\\", "/").lstrip("/")).as_posix()
        if final.as_posix() != original:
            renamed[original] = final.as_posix()
        written.append(_write_member(dest_dir, final, content))
        kept += len(content)
        bbl += 1
        files.append(final.as_posix())
```

### Size Limits, Path Sanitization & Atomic Cleanup
- **Member & Uncompressed Caps:** Individual members exceeding `MAX_MEMBER_BYTES` (100 MiB) or pushing total kept bytes past `MAX_UNCOMPRESSED_BYTES` (500 MiB) are skipped and recorded in `skipped_oversize_files` with `partial: true`.
- **Path Sanitization (`sanitize_relative_path`):** Strips leading slashes and `..` segments (preventing path traversal), replaces Windows-forbidden characters (`<>:"|?*` and control chars `< 32`) with `_`, and strips trailing spaces/dots.
- **Case-Insensitive Collision Handling (`_unique_rel`):** Prevents silent overwrites on Windows/macOS when two archive members differ only in case by appending `__dup1`, `__dup2`, and recording the `original -> stored` mapping in `renamed_files` for the downstream parser.
- **Atomic Rollback (`_cleanup`):** Any exception during extraction (including disk write errors `ExtractWriteError`) removes all files and newly created empty subdirectories written for that version.

## 2.3. Handling Divergent Source Packaging Types (`extractor.py`)

`extract_tex_and_bib_from_bytes()` inspects magic bytes in a strict decision cascade:

1. **HTTP 404 (`_NOT_FOUND_PAYLOAD`):** Returns `status = "no_tex_source"`, `source_type = "not_found_404"`.
2. **PDF-Only Submissions (`b"%PDF-"`):** Caught from the truncated first chunk without downloading or decompressing; returns `status = "no_tex_source"`, `source_type = "pdf_only"`.
3. **ZIP Archives (`b"PK\x03\x04"`):** Extracted in memory with `zipfile.ZipFile`. Corrupt archives return `error` (`corrupt_zip`); encrypted or unsupported compression returns `error` (`unsupported_zip`).
4. **Gzip Layer (`b"\x1f\x8b"`):** Decompressed via `_bounded_gunzip()` using `zlib.decompressobj(wbits=31)` with a 500 MiB cap and an explicit `d.eof` check to catch truncated streams (`corrupt_gzip`).
   - **Streamed-Tar Fallback (`_extract_big_gz` / `_extract_tar_stream`):** If a figure-heavy `.tar.gz` exceeds 500 MiB uncompressed, the extractor falls back to streaming mode (`tarfile.open(..., mode="r|gz")`) up to `MAX_TAR_TOTAL_BYTES = 4 GiB`, extracting wanted `.tex`/`.bib` members on the fly without holding the full decompressed tarball in RAM.
5. **Standard Tarballs (`tarfile.open(..., mode="r:*")`):** Unpacks multi-file LaTeX projects. If opening fails and bytes `[257:262] == b"ustar"`, reports `corrupt_tar`; otherwise falls through to single-file detection.
6. **Single Plain or Gzipped TeX/BibTeX File (`classify_plain_source`):** Inspects the first 64 KiB for LaTeX markers (`\documentclass`, `\begin{`, `\section`, etc.) or BibTeX markers (`@article`, `@inproceedings`, etc.), rejecting HTML and binaries. Valid single files are saved as `main.tex` or `refs.bib` (`source_type = "single_tex"` / `"single_bib"`), with `renamed_files = {"(single-file source)": fname}`.
7. **Unrecognized / Other Formats:** Returns `status = "no_tex_source"`, `source_type = "pdf_or_other"` (or `tar_without_tex` / `zip_without_tex` when an archive contained no `.tex`/`.bib`).

## 2.4. Two-Tier Concurrency, Memory Management & Checkpointing (`runner.py`, `pipeline.py`)

### Dual ThreadPool Architecture
Both `download_corpus()` (CLI batch mode) and `download_stream()` (live streaming mode fed by entry discovery) delegate to a single `_engine()`:
- **`net_pool` (`max_workers = connections`, default 8):** Runs `fetch_single_version_task` to overlap network latency while obeying the global rate slot.
- **`ext_pool` (`max_workers = max(4, workers)`, default 8):** Runs `extract_paper_to_existing_folders` to wait on a paper's version futures, unpack in memory, and write to disk.
- **Bounded Window:** At most `window = max(workers * 2, connections)` papers are in flight at once; `collect()` drains completed futures via `wait(..., return_when=FIRST_COMPLETED)`.

### Immediate RAM Release (`holder.pop()`)
To prevent completed `Future` objects from pinning large raw archives in RAM until garbage collection, `fetch_single_version_task()` wraps the payload in a one-element list `[data]`:

```python
# src/downloader/pipeline.py
ver_tag, full_vid, holder, err, fetch_sec = interruptible_result(fut)
...
raw_b = holder.pop()  # the Future's result no longer references the bytes
try:
    info = extract_tex_and_bib_from_bytes(raw_b, dest_dir=paper_dir / ver_tag)
finally:
    del raw_b
```

`interruptible_result()` polls `fut.result(timeout=1.0)` in a loop so `Ctrl+C` remains responsive on Windows.

### Atomic Checkpoints, Per-Version Resume & Circuit Breakers
- **Atomic Persistence:** Every `CHECKPOINT_EVERY = 20` completed papers (and on exit/interrupt), `download_status.json` is written compactly to `.tmp` and swapped with `os.replace()`. Corrupt checkpoints are moved aside to `.corrupt-<timestamp>` rather than silently overwritten.
- **Per-Version Resume (`_todo_versions`):** Only missing or retryable versions are fetched. Legacy `success` records lacking `bbl_file_count` are checked on disk (`_may_need_bbl`) and only re-fetched if no `.bib`, `.bbl`, or inline `\bibitem` exists.
- **Permanent vs. Transient Errors:** Permanent archive errors (`corrupt_gzip`, `corrupt_tar`, `corrupt_zip`, `unsupported_zip`, `gzip_too_large`) stop retrying after `MAX_VERSION_ATTEMPTS = 3` (`is_exhausted`). Network/5xx/403 errors remain retryable, and `InterruptedError` never increments `attempts`.
- **HTTP 403 Circuit Breaker:** After `MAX_BLOCKED_STREAK = 5` consecutive papers failing with `BlockedError`, `runner` sets `stop_event` and exits with code `3` (`reason = "blocked_by_server"`).
- **Graceful `SIGINT`/`SIGTERM`:** Stops in-flight retries, drains finished extractions, and flushes both `download_status.json` and `download_summary.json` (exit code `130`).

## 2.5. Post-Crawl Audit, Recovery & Execution Logging (`audit.py`, `stats.py`)

### Post-Crawl Audit & Recovery (`audit.py`)
Unless `--skip-audit` is passed, `run_audit()` scans all processed papers after the main loop:
1. **`find_missing_versions()`** flags any version whose record is still `is_retryable()` **or** whose target folder `<paper_dir>/<vN>` is missing on disk.
2. **`recover_missing_versions()`** re-fetches missing versions in parallel batches of `RECOVERY_BATCH = 16`, merges updated records via `merge_version_records()` + `refresh_record()`, and checkpoints after every batch.
3. **Audit Circuit Breaker:** Aborts recovery if `AUDIT_MAX_CONSECUTIVE_FAILURES = 10` consecutive recoveries fail, preserving all prior recovered work.

### Execution Audit & Summary Artifacts (`stats.py`)
All metadata is stored under `data/output/papers_download_meta/`:

1. **`download_status.json` (`SCHEMA_VERSION = 2`):** Per-paper record containing:
   - `arxiv_id`, `schema_version`, `paper_status` (`"success"`, `"no_tex_source"`, or `"error"`), `versions_expected`
   - `total_raw_archive_bytes`, `total_uncompressed_bytes`, `total_kept_tex_bib_bytes`, `download_time_sec`
   - `version_downloads[vN]`: `version_id`, `status`, `source_type`, `raw_archive_bytes`, `uncompressed_total_bytes`, `kept_tex_bib_bytes`, `tex_file_count`, `bib_file_count`, `bbl_file_count`, `kept_files`, plus optional `skipped_oversize_files`, `partial`, `renamed_files`, `attempts`, and `error`.
2. **`download_summary.json`** (or `download_summary_limit.json` when `--limit` is used):
   - Cumulative timing across resumed sessions (`previous_wall_time_sec`, `session_wall_time_sec`, `total_wall_time_sec`, `avg_wall_time_per_paper_sec`).
   - OS-level Peak RAM (`peak_rss_mb` via `resource.getrusage` on POSIX or `psutil` `peak_wset` on Windows, capturing C-level `zlib`/`tarfile` buffers).
   - Storage efficiency (`compute_storage_metrics`): `total_raw_archive_bytes`, `total_uncompressed_bytes`, `total_kept_tex_bib_bytes`, `total_saved_bytes_vs_uncompressed`, `storage_reduction_ratio_vs_uncompressed_pct`, `storage_reduction_ratio_vs_raw_archive_pct`, `versions_by_source_type`, and counts of 404, partial, renamed, conditional `.bbl`, and permanently failed versions.

---
# Phiên bản Tiếng Việt — Module 2: Tải Mã nguồn Toàn văn & Giải nén Chọn lọc (`src/downloader`)

## Tổng quan & Kiến trúc Module

Gói `src/downloader` chịu trách nhiệm tải gói mã nguồn LaTeX của từng cặp `(arxiv_id, version)` từ arXiv, kiểm tra và giải nén **hoàn toàn trong bộ nhớ (in-memory)**, chỉ lưu xuống đĩa các tệp văn bản cần thiết cho bước phân tích cú pháp và trích xuất tài liệu tham khảo phía sau (`.tex`, `.bib`, và `.bbl` có điều kiện). Các tài nguyên nhị phân nặng (ảnh, PDF, tệp biên dịch trung gian) không bao giờ được ghi xuống đĩa.

| Module | Vai trò chính |
| :--- | :--- |
| `config.py` | Khai báo hằng số hệ thống: endpoint arXiv, phần mở rộng cho phép/có điều kiện, trần RAM & dung lượng tệp, số lần retry, ngưỡng circuit breaker, đường dẫn mặc định. |
| `index.py` | Đọc và kiểm tra hợp lệ `data/input/papers_index.json` (`{arxiv_id: ["v1", "v2", ...]}`), bắt buộc ID dạng mới (`YYMM.NNNNN`) và nhãn `vN` hợp lệ. |
| `client.py` | HTTP client đa luồng an toàn (`FastArxivClient`) với Keep-Alive theo từng luồng, khóa điều tiết tốc độ toàn cục, làm mát thích ứng (adaptive cooldown), cắt sớm PDF-only và cơ chế retry. |
| `extractor.py` | Kiểm tra và giải nén chọn lọc trong RAM (`.tar`, `.tar.gz`, streamed `.tar.gz`, `.zip`, tệp đơn `.tex`/`.bib`), làm sạch đường dẫn, đổi tên khi trùng, giữ `.bbl` có điều kiện và dọn dẹp nguyên tử khi lỗi. |
| `utils.py` | Các hàm tiện ích không phụ thuộc ngoài: nhận diện TeX/Bib/HTML, phát hiện `\bibitem` bỏ qua comment `%`, làm sạch đường dẫn, định dạng dung lượng/thời gian, đo Peak RSS mức hệ điều hành. |
| `pipeline.py` | Đơn vị công việc theo từng bài báo: tải đồng thời các version, giải nén vào `papers/<YYMM-NNNNN>/<vN>/`, đếm số lần thử lỗi, hợp nhất trạng thái và cộng tổng dung lượng. |
| `runner.py` | Bộ điều phối lõi (`_engine`) dùng chung cho cả chế độ batch (`download_corpus`) và streaming (`download_stream`), quản lý hai thread pool, checkpoint nguyên tử, resume theo từng version, bắt tín hiệu ngắt và cầu chì HTTP 403. |
| `audit.py` | Hậu kiểm sau khi tải (`run_audit`) để phát hiện và tải bù các version bị thiếu hoặc lỗi tạm thời theo từng lô song song có giới hạn. |
| `stats.py` | Tổng hợp chỉ số lưu trữ toàn tập dữ liệu, ghi nguyên tử `download_summary.json` và in báo cáo console. |
| `cli.py` / `__main__.py` | Giao diện dòng lệnh (`python -m src.downloader`). |

## 2.1. Luồng Tải theo Phiên bản, Điều tiết Tốc độ & Chịu lỗi (`client.py`, `index.py`)

### Kiểm tra đầu vào (`index.py`)
`load_papers_index()` chỉ đọc mà không chỉnh sửa `papers_index.json`, đồng thời kiểm tra nghiêm ngặt:
- `arxiv_id` phải khớp `^\d{4}\.\d{4,5}$` (từ chối ID cũ dạng `hep-th/9901001` để tránh dấu `/` tạo thư mục lồng nhau sai cấu trúc khi đổi `.` thành `-`).
- Danh sách version không được rỗng và mọi phần tử khớp `^v\d+$`.

### HTTP Client & Điều tiết tốc độ toàn cục (`client.py`)
`FastArxivClient` cấp cho mỗi luồng một `requests.Session` riêng qua `threading.local()` (`HTTPAdapter(pool_connections=2, pool_maxsize=2)`, `Connection: keep-alive`) kèm `User-Agent` cố định, trung thực (có hỗ trợ thêm `--contact-email`). Dù chạy nhiều luồng tải song song (`DEFAULT_CONNECTIONS = 8`), mọi request đều phải đi qua cổng điều tiết toàn cục `_wait_rate_slot()`:

```python
# src/downloader/client.py
def _wait_rate_slot(self) -> int:
    while True:
        if self.stop_event is not None and self.stop_event.is_set():
            raise InterruptedError('Pipeline stopped.')
        with self._rate_lock:
            now = time.monotonic()
            with self._cooldown_lock:
                pacing = self.delay - (now - self._last_req_ts) if self._last_req_ts is not None else 0.0
                wait = max(0.0, self._cooldown_until - now, pacing)
            if self.shared_cooldown is not None:
                wait = max(wait, self.shared_cooldown.remaining())
            if wait <= 0:
                idx = self._counter
                self._counter += 1
                self._last_req_ts = now
                return idx
        self._sleep(min(wait, 2.0))
```

Các cơ chế đảm bảo độ tin cậy trong `fetch_version()`:
1. **Endpoint dự phòng:** Gọi `https://export.arxiv.org/src/{id}v{n}` trước và luân phiên sang `https://export.arxiv.org/e-print/{id}v{n}` khi retry (tối đa `MAX_FETCH_ATTEMPTS = 7` lần).
2. **Làm mát thích ứng (Adaptive Cooldown):** Khi gặp HTTP `429` hoặc `403`, `_note_throttle()` đặt mốc `_cooldown_until` tối thiểu `max(min(25, 10 + 5 * attempt), Retry-After)` (chặn trên ở `MAX_RETRY_AFTER_SEC = 300s`), tăng `delay` lên `1.25x` (không vượt `max(MAX_ADAPTIVE_DELAY, 2 * base_delay)`), và phát `429` lên `shared_cooldown` để các module gọi arXiv khác cùng tạm dừng. Sau 25 request sạch liên tiếp, `_note_success()` giảm `delay` theo hệ số `0.8x` về lại `base_delay` (`DEFAULT_DELAY = 0.6s`).
3. **Phát hiện chặn HTTP 403:** Nếu `403` lặp lại `MAX_403_ATTEMPTS = 2` lần trên cùng version, client lập tức ném `BlockedError` mà không retry thêm.
4. **Phân biệt HTTP 404:** Trả về hằng số đánh dấu `_NOT_FOUND_PAYLOAD` thay vì retry, giúp tách bạch bản ghi 404 với bài báo chỉ có PDF.
5. **Đọc luồng & Cắt sớm bài PDF-only (`_read_body`):** Đọc từng khối 64 KiB, chặn trên `MAX_ARCHIVE_BYTES = 500 MiB` (`ArchiveTooLarge`, không retry), **ngắt ngay ở khối đầu tiên nếu bắt đầu bằng `b"%PDF-"`** (tránh tải cả tệp PDF lớn), và retry (`_BadPayload`) nếu thân phản hồi rỗng, là trang lỗi HTML (`looks_like_html`), hoặc bị cắt cụt (`len(body) != Content-Length`).

## 2.2. Cơ chế Giải nén Chọn lọc & An toàn Đường dẫn (`extractor.py`, `utils.py`)

Toàn bộ quá trình kiểm tra gói lưu trữ diễn ra trong RAM qua `io.BytesIO`; chỉ các tệp văn bản đạt tiêu chuẩn mới được ghi vào `data/output/papers/<YYMM-NNNNN>/<vN>/`.

### Luật lọc tệp & Giữ `.bbl` có điều kiện
- **Luôn giữ lại (`ALLOWED_EXTENSIONS`):** Các tệp `.tex` và `.bib` (không phân biệt hoa/thường).
- **Loại bỏ ngay trong RAM:** Ảnh (`.png`, `.jpg`, `.eps`, `.pdf`), tệp biên dịch trung gian (`.aux`, `.log`, `.out`, `.toc`), tệp rác macOS (`__MACOSX`, `._*`), tệp rỗng/chỉ chứa khoảng trắng, và tệp nhị phân đội lốt văn bản (chứa byte `\x00` trong 8 KiB đầu hoặc mang magic bytes nhị phân).
- **Giữ `.bbl` có điều kiện (`CONDITIONAL_EXTENSIONS`):** Các tệp `.bbl` được tạm giữ trong `pending_bbl` khi duyệt archive và **chỉ ghi xuống đĩa khi** version có ít nhất 1 tệp `.tex`, **không có tệp `.bib` nào**, và **không có lệnh `\bibitem` thực sự** (bỏ qua phần sau `%` comment) trong các tệp `.tex` đã giữ:

```python
# src/downloader/extractor.py
if tex > 0 and bib == 0 and not has_bibitem:
    skipped.extend(oversize_bbl)  # a missing .bbl matters only now
    for name, rel, content in pending_bbl:
        final = _unique_rel(seen, rel)
        original = PurePosixPath(name.replace("\\", "/").lstrip("/")).as_posix()
        if final.as_posix() != original:
            renamed[original] = final.as_posix()
        written.append(_write_member(dest_dir, final, content))
        kept += len(content)
        bbl += 1
        files.append(final.as_posix())
```

### Giới hạn dung lượng, Làm sạch đường dẫn & Dọn dẹp nguyên tử
- **Trần dung lượng:** Tệp đơn vượt `MAX_MEMBER_BYTES` (100 MiB) hoặc làm tổng dung lượng giữ lại vượt `MAX_UNCOMPRESSED_BYTES` (500 MiB) sẽ bị bỏ qua, ghi nhận vào `skipped_oversize_files` kèm cờ `partial: true`.
- **Làm sạch đường dẫn (`sanitize_relative_path`):** Bỏ dấu `/` đầu và các đoạn `..` (chống path traversal), thay ký tự cấm trên Windows (`<>:"|?*` và ký tự điều khiển `< 32`) bằng `_`, cắt khoảng trắng/dấu chấm ở cuối tên.
- **Chống ghi đè trùng tên (`_unique_rel`):** So sánh đường dẫn không phân biệt hoa/thường để tránh đè tệp trên Windows/macOS; khi trùng sẽ tự thêm hậu tố `__dup1`, `__dup2` và lưu ánh xạ `original -> stored` vào `renamed_files` cho bước parser phía sau.
- **Hoàn tác nguyên tử (`_cleanup`):** Mọi ngoại lệ trong lúc giải nén (kể cả lỗi ghi đĩa `ExtractWriteError`) sẽ xóa sạch toàn bộ tệp và thư mục con rỗng mới tạo của version đó để không để lại dữ liệu dở dang.

## 2.3. Xử lý Đa dạng Định dạng Đóng gói Mã nguồn (`extractor.py`)

Hàm `extract_tex_and_bib_from_bytes()` phân loại payload theo thứ tự kiểm tra magic bytes:

1. **HTTP 404 (`_NOT_FOUND_PAYLOAD`):** Trả về `status = "no_tex_source"`, `source_type = "not_found_404"`.
2. **Bài nộp chỉ có PDF (`b"%PDF-"`):** Phát hiện ngay từ khối dữ liệu đầu tiên bị cắt sớm, bỏ qua giải nén và trả về `status = "no_tex_source"`, `source_type = "pdf_only"`.
3. **Gói ZIP (`b"PK\x03\x04"`):** Mở trực tiếp trong RAM bằng `zipfile.ZipFile`. Gói hỏng trả về `error` (`corrupt_zip`); gói mã hóa hoặc dùng thuật toán nén không hỗ trợ trả về `error` (`unsupported_zip`).
4. **Lớp nén Gzip (`b"\x1f\x8b"`):** Giải nén có giới hạn bằng `_bounded_gunzip()` (`zlib.decompressobj(wbits=31)`, trần 500 MiB) và kiểm tra `d.eof` để bắt luồng bị cắt cụt (`corrupt_gzip`).
   - **Dự phòng đọc luồng Streamed Tar (`_extract_big_gz` / `_extract_tar_stream`):** Với các gói `.tar.gz` chứa nhiều hình ảnh khiến kích thước sau giải nén vượt 500 MiB, hệ thống tự chuyển sang đọc tuần tự bằng `tarfile.open(..., mode="r|gz")` (trần `MAX_TAR_TOTAL_BYTES = 4 GiB`), trích ngay `.tex`/`.bib` mà không nạp toàn bộ tarball đã giải nén vào RAM.
5. **Gói Tar chuẩn (`tarfile.open(..., mode="r:*")`):** Giải nén dự án LaTeX nhiều tệp. Nếu lỗi mà byte `[257:262] == b"ustar"` thì báo `corrupt_tar`; ngược lại chuyển xuống bước kiểm tra tệp đơn.
6. **Tệp văn bản đơn `.tex` / `.bib` (`classify_plain_source`):** Quét 64 KiB đầu tìm từ khóa LaTeX (`\documentclass`, `\begin{`, `\section`, ...) hoặc BibTeX (`@article`, `@inproceedings`, ...), loại trừ HTML và nhị phân. Tệp hợp lệ được lưu thành `main.tex` hoặc `refs.bib` (`source_type = "single_tex"` / `"single_bib"`), ghi nhận `renamed_files = {"(single-file source)": fname}`.
7. **Định dạng khác / Không có TeX:** Trả về `status = "no_tex_source"`, `source_type = "pdf_or_other"` (hoặc `tar_without_tex` / `zip_without_tex` nếu gói lưu trữ không chứa `.tex`/`.bib`).

## 2.4. Xử lý Song song Hai tầng, Quản lý RAM & Checkpoint (`runner.py`, `pipeline.py`)

### Kiến trúc 2 ThreadPool tách biệt
Cả `download_corpus()` (chạy batch từ CLI) và `download_stream()` (chạy song song theo luồng bài báo từ bước discovery) đều dùng chung bộ điều phối `_engine()`:
- **`net_pool` (`max_workers = connections`, mặc định 8):** Chạy `fetch_single_version_task` để ẩn độ trễ mạng trong khi vẫn tuân thủ khoảng giãn cách toàn cục.
- **`ext_pool` (`max_workers = max(4, workers)`, mặc định 8):** Chạy `extract_paper_to_existing_folders` để chờ các `Future` của bài báo, giải nén trong RAM và ghi đĩa.
- **Cửa sổ trượt (Backpressure):** Khống chế tối đa `window = max(workers * 2, connections)` bài báo đang xử lý cùng lúc; khi đạt ngưỡng sẽ gọi `collect()` (`wait(..., return_when=FIRST_COMPLETED)`) để thu gom trước khi đẩy thêm.

### Giải phóng RAM tức thì khỏi `Future` (`holder.pop()`)
Để tránh các đối tượng `Future` đã xong tiếp tục giữ tham chiếu tới gói archive lớn trong RAM, `fetch_single_version_task()` bọc `bytes` trong list 1 phần tử `[data]`:

```python
# src/downloader/pipeline.py
ver_tag, full_vid, holder, err, fetch_sec = interruptible_result(fut)
...
raw_b = holder.pop()  # Future không còn giữ tham chiếu tới bộ đệm bytes
try:
    info = extract_tex_and_bib_from_bytes(raw_b, dest_dir=paper_dir / ver_tag)
finally:
    del raw_b
```

Hàm `interruptible_result()` gọi `fut.result(timeout=1.0)` theo vòng lặp để phím `Ctrl+C` trên Windows luôn cắt được ngay lập tức.

### Checkpoint nguyên tử, Resume theo phiên bản & Cầu chì bảo vệ
- **Ghi Checkpoint nguyên tử:** Sau mỗi `CHECKPOINT_EVERY = 20` bài hoàn thành (và khi kết thúc/bị ngắt), `download_status.json` được ghi ra tệp `.tmp` dạng JSON nén rồi thay thế bằng `os.replace()`. Tệp checkpoint hỏng sẽ được đổi tên sang `.corrupt-<timestamp>` thay vì im lặng ghi đè.
- **Resume ở mức version (`_todo_versions`):** Chỉ tải những version chưa xong và còn lượt thử. Các bản ghi `success` cũ chưa có `bbl_file_count` được kiểm tra trên đĩa (`_may_need_bbl`) và chỉ tải lại nếu thiếu cả `.bib`, `.bbl` lẫn `\bibitem`.
- **Phân biệt lỗi vĩnh viễn và lỗi tạm thời:** Các lỗi gói hỏng thuộc `PERMANENT_ERROR_TYPES` (`corrupt_gzip`, `corrupt_tar`, `corrupt_zip`, `unsupported_zip`, `gzip_too_large`) sẽ ngừng retry sau `MAX_VERSION_ATTEMPTS = 3` lần (`is_exhausted`). Lỗi mạng/5xx/403 không bị khóa vĩnh viễn; lỗi `InterruptedError` khi dừng chương trình không làm tăng `attempts`.
- **Cầu chì HTTP 403:** Nếu `MAX_BLOCKED_STREAK = 5` bài liên tiếp gặp `BlockedError`, hệ thống kích hoạt `stop_event` và dừng với mã thoát `3` (`reason = "blocked_by_server"`).
- **Xử lý `SIGINT`/`SIGTERM`:** Dừng sớm các luồng mạng đang retry, thu gom các bài đã giải nén xong rồi lưu đầy đủ `download_status.json` và `download_summary.json` (mã thoát `130`).

## 2.5. Hậu kiểm, Phục hồi & Ghi nhận Báo cáo Thực thi (`audit.py`, `stats.py`)

### Hậu kiểm và Phục hồi phiên bản thiếu (`audit.py`)
Sau khi hoàn tất vòng tải chính (trừ khi truyền `--skip-audit`), `run_audit()` quét lại toàn bộ danh sách đã xử lý:
1. **`find_missing_versions()`** đánh dấu version cần phục hồi nếu bản ghi còn `is_retryable()` **hoặc** thư mục `<paper_dir>/<vN>` không tồn tại trên đĩa.
2. **`recover_missing_versions()`** tải bù theo từng lô `RECOVERY_BATCH = 16` version song song qua `net_pool`, hợp nhất kết quả bằng `merge_version_records()` + `refresh_record()` và gọi `checkpoint()` sau mỗi lô.
3. **Cầu chì dừng sớm Audit:** Nếu `AUDIT_MAX_CONSECUTIVE_FAILURES = 10` lần phục hồi liên tiếp đều thất bại, audit lập tức dừng để tránh lặp vô ích khi mất mạng/bị chặn, đồng thời giữ lại toàn bộ phần đã phục hồi.

### Tệp Nhật ký Kiểm toán & Thống kê (`stats.py`)
Toàn bộ metadata được lưu tại `data/output/papers_download_meta/`:

1. **`download_status.json` (`SCHEMA_VERSION = 2`):** Bản ghi theo từng bài báo gồm:
   - `arxiv_id`, `schema_version`, `paper_status` (`"success"`, `"no_tex_source"`, hoặc `"error"`), `versions_expected`
   - `total_raw_archive_bytes`, `total_uncompressed_bytes`, `total_kept_tex_bib_bytes`, `download_time_sec`
   - `version_downloads[vN]`: `version_id`, `status`, `source_type`, `raw_archive_bytes`, `uncompressed_total_bytes`, `kept_tex_bib_bytes`, `tex_file_count`, `bib_file_count`, `bbl_file_count`, `kept_files`, cùng các trường tùy chọn `skipped_oversize_files`, `partial`, `renamed_files`, `attempts`, và `error`.
2. **`download_summary.json`** (hoặc `download_summary_limit.json` khi chạy với `--limit`):
   - Thời gian cộng dồn qua các phiên chạy resume (`previous_wall_time_sec`, `session_wall_time_sec`, `total_wall_time_sec`, `avg_wall_time_per_paper_sec`).
   - Bộ nhớ đỉnh thực tế mức hệ điều hành (`peak_rss_mb` qua `resource.getrusage` trên POSIX hoặc `psutil` `peak_wset` trên Windows, phản ánh cả bộ đệm C của `zlib`/`tarfile`).
   - Hiệu quả lưu trữ (`compute_storage_metrics`): `total_raw_archive_bytes`, `total_uncompressed_bytes`, `total_kept_tex_bib_bytes`, `total_saved_bytes_vs_uncompressed`, `storage_reduction_ratio_vs_uncompressed_pct`, `storage_reduction_ratio_vs_raw_archive_pct`, phân bố `versions_by_source_type`, và thống kê số version 404, bị cắt bớt do quá cỡ (`partial`), có đổi tên tệp, số `.bbl` giữ có điều kiện và số version lỗi vĩnh viễn.